In [1]:
import os
import random
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import Dict

/home/airlay88/planscape/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class LocalLLM:
    def __init__(self, model_id: str, device: str = 'cuda'):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map='auto',
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        self.device = self.model.device

    def __call__(self, prompt: str, **generate_kwargs) -> str:
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.device)
        defaults = dict(max_new_tokens=256, do_sample=False)
        params = {**defaults, **generate_kwargs}
        with torch.no_grad():
            out = self.model.generate(**inputs, **params)
        text = self.tokenizer.decode(out[0], skip_special_tokens=True)
        return text[len(prompt):].strip()

# Cache and model mapping
LLM_CACHE: Dict[str, LocalLLM] = {}
MODEL_MAP = {
    'qwen2.5': 'Qwen/Qwen2.5-7B-Instruct',
    'llama3.1': 'meta-llama/Llama-3.1-8B-Instruct',
    'gemma2': 'google/gemma-2-9b-it'
}


In [3]:

def get_llm(key: str) -> LocalLLM:
    if key not in LLM_CACHE:
        model_id = MODEL_MAP[key]
        LLM_CACHE[key] = LocalLLM(model_id)
    return LLM_CACHE[key]

In [4]:
# Configuration
root_dir = 'experiment_clusters'
templates_dir = './'
output_dir = './output'
model_key = 'llama3.1'  # choose from MODEL_MAP

group_size = 5

In [5]:

with open("eval_prompts.json", "r") as f:
    eval_prompts = json.load(f)


In [6]:
results = []
llm = get_llm(model_key)
for itype, scenarios in eval_prompts.items():
    print(itype)
    for scenario, items in scenarios.items():
        print(scenario)
        for entry in items:
            print(entry)
            response = llm(entry['prompt'], temperature=0.0)
            # assume first token/ID is the predicted plan ID
            lines = response.splitlines()
            pred_id = lines[0].strip()
            explanation = '\n'.join(lines[1:]).strip()
            results.append({
                'model': model_key,
                'input_type': itype,
                'scenario': scenario,
                'outlier_id': entry['outlier_id'],
                'predicted_id': pred_id,
                'explanation': explanation,
                'raw_response': response
            })

# Save results
with open(os.path.join(output_dir, f'llm_results_{model_key}.json'), 'w') as f:
    json.dump(results, f, indent=2)


Fetching 4 files:   0%|          | 0/4 [00:11<?, ?it/s]


KeyboardInterrupt: 